## 0. 加载 Qwen 模型

本 Notebook 使用 **Qwen2.5-7B-Instruct**（通过 ModelScope 加载）作为真实 LLM 后端，替代原有的模拟 LLM。

> **低显存备选**：如果 GPU 显存不足，可将模型 ID 替换为 `Qwen/Qwen2.5-3B-Instruct`。

### 安装依赖（首次运行需要）

```bash
pip install modelscope torch transformers
```


In [ ]:
# ============================================================
# 加载 Qwen 模型（通过 ModelScope）
# ============================================================
# 如果没有 GPU 或显存不足，可将模型 ID 改为 Qwen/Qwen2.5-3B-Instruct
# ============================================================

import torch
from modelscope import AutoModelForCausalLM, AutoTokenizer


class QwenLLM:
    """基于 ModelScope 的 Qwen 模型封装类"""

    def __init__(self, model_name="Qwen/Qwen2.5-7B-Instruct", device=None):
        # 自动检测 GPU / CPU
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device
        print(f"[QwenLLM] 使用设备: {self.device}")
        print(f"[QwenLLM] 加载模型: {model_name}")

        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype="auto", device_map="auto"
        )
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.messages = []
        print("[QwenLLM] 模型加载完成！")

    def chat(self, user_message, system_prompt=None, max_new_tokens=512, temperature=0.7):
        """调用模型进行对话"""
        if system_prompt:
            messages = [{"role": "system", "content": system_prompt}]
        else:
            messages = []
        messages.extend(self.messages)
        messages.append({"role": "user", "content": user_message})

        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)

        generated_ids = self.model.generate(
            **model_inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=True
        )
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        self.messages.append({"role": "user", "content": user_message})
        self.messages.append({"role": "assistant", "content": response})
        return response

    def reset(self):
        """重置对话历史"""
        self.messages = []


# 初始化 QwenLLM 实例
llm = QwenLLM(model_name="Qwen/Qwen2.5-7B-Instruct")
print("\n模型就绪，可以开始使用了。")

# 项目一：知识库问答系统

## 项目目标

构建一个基于 RAG 的知识库问答系统，能够：
- 加载和索引文档
- 回答基于知识库的问题
- 支持多轮对话
- 提供引用来源

## 技术栈

- 文档处理：文本分块
- 向量检索：相似度搜索
- 对话管理：上下文维护
- 回答生成：基于检索结果

---

## 1. 系统架构

```
┌─────────────────────────────────────────────────────────────┐
│                  知识库问答系统                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌─────────────┐    ┌─────────────┐    ┌─────────────┐     │
│  │  文档加载    │───►│  文本分块    │───►│  向量索引    │     │
│  └─────────────┘    └─────────────┘    └──────┬──────┘     │
│                                                │            │
│  ┌─────────────┐    ┌─────────────┐    ┌──────┴──────┐     │
│  │  回答生成    │◄───│  上下文构建  │◄───│  相似度检索  │     │
│  └─────────────┘    └─────────────┘    └─────────────┘     │
│         │                                                   │
│         ▼                                                   │
│  ┌─────────────┐                                            │
│  │  用户界面    │                                            │
│  └─────────────┘                                            │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

---

## 2. 实现代码

In [ ]:
import numpy as np
from typing import List, Dict, Optional
import re

# 知识库数据
KNOWLEDGE_BASE = {
    "ai_agent": """
    AI Agent（人工智能代理）是一种能够感知环境、做出决策并执行动作的智能系统。
    核心特征包括：自主性（Autonomy）、反应性（Reactivity）、主动性（Pro-activeness）和社会性（Social Ability）。
    常见的 AI Agent 架构包括：ReAct、Reflexion、AutoGPT、BabyAGI 等。
    """,
    "react": """
    ReAct（Reasoning + Acting）是一种将推理和行动结合的 Agent 架构。
    它通过交替进行思考（Thought）和行动（Action）来解决复杂问题。
    ReAct 的优势在于能够利用外部工具获取信息，并基于这些信息进行推理。
    典型流程：Thought -> Action -> Observation -> Thought -> ...
    """,
    "langchain": """
    LangChain 是一个用于开发 LLM 应用的框架，提供了：
    1. 模型接口（Models）：支持多种 LLM 和聊天模型
    2. 提示词管理（Prompts）：模板化和动态构建提示词
    3. 链式调用（Chains）：将多个组件组合成流水线
    4. 记忆系统（Memory）：维护对话上下文
    5. Agent 系统：支持工具使用和自主决策
    """,
    "rag": """
    RAG（Retrieval-Augmented Generation）是一种将信息检索与文本生成结合的技术。
    工作流程：文档加载 -> 文本分块 -> 向量化 -> 存储 -> 检索 -> 生成回答。
    RAG 的优势：减少幻觉、支持实时知识更新、提供可解释性。
    """,
    "vector_db": """
    向量数据库用于存储和检索高维向量，是 RAG 系统的核心组件。
    常见选择：Chroma（轻量）、FAISS（高效）、Pinecone（托管）、Weaviate（功能丰富）。
    核心操作：添加向量、相似度搜索、元数据过滤。
    """
}

class SimpleKnowledgeBaseQA:
    """知识库问答系统（使用 QwenLLM 进行回答生成）"""
    
    def __init__(self, llm_model=None):
        self.documents = []
        self.vectors = []
        self.dim = 64
        self.conversation_history = []
        self.llm = llm_model  # QwenLLM 实例
    
    def _text_to_vector(self, text: str) -> np.ndarray:
        """文本向量化"""
        words = text.lower().split()
        vector = np.zeros(self.dim)
        for word in words:
            idx = hash(word) % self.dim
            vector[idx] += 1.0
        norm = np.linalg.norm(vector)
        return vector / norm if norm > 0 else vector
    
    def load_knowledge_base(self, kb: Dict[str, str]):
        """加载知识库"""
        for topic, content in kb.items():
            chunks = self._split_text(content)
            for chunk in chunks:
                doc = {
                    "content": chunk,
                    "topic": topic,
                    "vector": self._text_to_vector(chunk)
                }
                self.documents.append(doc)
        
        print(f"已加载 {len(kb)} 个主题，共 {len(self.documents)} 个文档块")
    
    def _split_text(self, text: str, chunk_size: int = 100) -> List[str]:
        """文本分块"""
        sentences = re.split(r'(?<=[。！？.!?])\s+', text.strip())
        chunks = []
        current = ""
        
        for sent in sentences:
            if len(current) + len(sent) <= chunk_size:
                current += sent
            else:
                if current:
                    chunks.append(current)
                current = sent
        
        if current:
            chunks.append(current)
        
        return chunks
    
    def retrieve(self, query: str, top_k: int = 3) -> List[Dict]:
        """检索相关文档"""
        query_vec = self._text_to_vector(query)
        
        scores = []
        for doc in self.documents:
            score = np.dot(query_vec, doc["vector"])
            scores.append((doc, score))
        
        scores.sort(key=lambda x: x[1], reverse=True)
        return [doc for doc, score in scores[:top_k] if score > 0]
    
    def answer(self, question: str) -> Dict:
        """回答问题（使用 QwenLLM）"""
        # 检索
        docs = self.retrieve(question)
        
        if not docs:
            return {
                "question": question,
                "answer": "抱歉，我在知识库中没有找到相关信息。",
                "sources": []
            }
        
        # 构建上下文
        context = "\n".join([d["content"] for d in docs])
        
        if self.llm is not None:
            # 使用 QwenLLM 生成回答
            system_prompt = (
                "你是一个知识库问答助手。请根据提供的参考文档内容回答用户问题。"
                "回答要准确、简洁。"
            )
            user_msg = f"参考文档：\n{context}\n\n用户问题：{question}"
            answer = self.llm.chat(user_msg, system_prompt=system_prompt, max_new_tokens=512, temperature=0.7)
        else:
            # ---- 无模型时的备选方案 ----
            answer = f"基于知识库信息：\n\n{context[:300]}..."
        
        result = {
            "question": question,
            "answer": answer,
            "sources": [d["topic"] for d in docs]
        }
        
        self.conversation_history.append(result)
        return result

# 创建并测试系统（传入 QwenLLM 实例）
qa_system = SimpleKnowledgeBaseQA(llm_model=llm)
qa_system.load_knowledge_base(KNOWLEDGE_BASE)

# 测试
questions = [
    "什么是 AI Agent?",
    "ReAct 是什么?",
    "LangChain 有什么特点?"
]

for q in questions:
    result = qa_system.answer(q)
    print(f"\nQ: {result['question']}")
    print(f"A: {result['answer'][:200]}...")
    print(f"来源: {result['sources']}")

---

## 3. 系统扩展

### 3.1 添加对话记忆

```python
class ConversationalKBQA(SimpleKnowledgeBaseQA):
    """支持多轮对话的知识库问答"""
    
    def __init__(self):
        super().__init__()
        self.session_context = []
    
    def answer(self, question: str) -> Dict:
        # 结合历史上下文
        enriched_query = self._enrich_query(question)
        return super().answer(enriched_query)
    
    def _enrich_query(self, question: str) -> str:
        """结合历史上下文丰富查询"""
        if not self.session_context:
            return question
        
        context = " ".join([item["question"] for item in self.session_context[-3:]])
        return f"{context} {question}"
```

### 3.2 添加反馈机制

```python
def provide_feedback(self, question: str, helpful: bool):
    """用户反馈"""
    for item in self.conversation_history:
        if item["question"] == question:
            item["feedback"] = helpful
            break
```

---

## 4. 项目总结

### 实现的功能

1. ✅ 文档加载和分块
2. ✅ 向量化和相似度检索
3. ✅ 基于检索结果的回答生成
4. ✅ 引用来源展示

### 可优化方向

- 使用真实的嵌入模型（如 OpenAI Embedding）
- 集成真实的向量数据库
- 添加重排序机制
- 支持多模态数据
- 添加用户反馈循环

---

## 参考

- [LangChain RAG Tutorial](https://python.langchain.com/docs/use_cases/question_answering/)
- [RAG Best Practices](https://www.pinecone.io/learn/retrieval-augmented-generation/)